In [1]:
!nvidia-smi

Tue Jul 14 19:26:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import tensorflow as tf

print(tf.__version__)
print(tf.config.list_physical_devices('GPU'))

2.20.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"junaidymuhtarom1927","key":"70c022d2495b8442cad9bb86af1ae64a"}'}

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os

SAVE_DIR = "/content/drive/MyDrive/MoodifyAI"

os.makedirs(SAVE_DIR, exist_ok=True)

print("Folder siap:", SAVE_DIR)

Folder siap: /content/drive/MyDrive/MoodifyAI


In [6]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [7]:
!kaggle datasets download -d msambare/fer2013

Dataset URL: https://www.kaggle.com/datasets/msambare/fer2013
License(s): DbCL-1.0
100% 60.3M/60.3M [00:00<00:00, 143MB/s]



In [8]:
!unzip -q fer2013.zip -d dataset

In [9]:
!find dataset -type d | head -20

dataset
dataset/test
dataset/test/angry
dataset/test/disgust
dataset/test/happy
dataset/test/surprise
dataset/test/neutral
dataset/test/fear
dataset/test/sad
dataset/train
dataset/train/angry
dataset/train/disgust
dataset/train/happy
dataset/train/surprise
dataset/train/neutral
dataset/train/fear
dataset/train/sad


In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
import os

SAVE_DIR="/content/drive/MyDrive/MoodifyAI"

os.makedirs(SAVE_DIR,exist_ok=True)

print(SAVE_DIR)

/content/drive/MyDrive/MoodifyAI


In [12]:
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator

from tensorflow.keras.applications import MobileNetV2

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import GlobalAveragePooling2D

from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.callbacks import ModelCheckpoint

In [13]:
IMG_SIZE=224
BATCH_SIZE=32

train_datagen=ImageDataGenerator(

rescale=1./255,

rotation_range=15,

zoom_range=0.15,

width_shift_range=0.1,

height_shift_range=0.1,

horizontal_flip=True

)

test_datagen=ImageDataGenerator(
rescale=1./255
)

In [14]:
train_generator=train_datagen.flow_from_directory(

"dataset/train",

target_size=(224,224),

batch_size=BATCH_SIZE,

class_mode="categorical",

shuffle=True

)

test_generator=test_datagen.flow_from_directory(

"dataset/test",

target_size=(224,224),

batch_size=BATCH_SIZE,

class_mode="categorical",

shuffle=False

)

Found 28709 images belonging to 7 classes.
Found 7178 images belonging to 7 classes.


In [15]:
print(train_generator.class_indices)

{'angry': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 'neutral': 4, 'sad': 5, 'surprise': 6}


In [20]:
from tensorflow.keras.applications import MobileNetV2

base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

# Bekukan sebagian besar layer
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Buka 30 layer terakhir
for layer in base_model.layers[-30:]:
    layer.trainable = True

In [21]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *

model = Sequential([

    base_model,

    GlobalAveragePooling2D(),

    BatchNormalization(),

    Dense(
        512,
        activation="relu"
    ),

    Dropout(0.5),

    Dense(
        256,
        activation="relu"
    ),

    Dropout(0.3),

    Dense(
        7,
        activation="softmax"
    )

])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 512)            │       655,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,052,103 (11.64 MB)

 Trainable params: 2,317,959 (8.84 MB)

 Non-trainable params: 734,144 (2.80 MB)

In [22]:
from tensorflow.keras.optimizers import Adam

model.compile(

    optimizer=Adam(
        learning_rate=1e-4
    ),

    loss=tf.keras.losses.CategoricalCrossentropy(
        label_smoothing=0.1
    ),

    metrics=["accuracy"]

)

In [23]:
checkpoint = ModelCheckpoint(

"/content/drive/MyDrive/MoodifyAI/best_model.keras",

monitor="val_accuracy",

save_best_only=True,

mode="max",

verbose=1

)

earlystop = EarlyStopping(

monitor="val_accuracy",

patience=5,

restore_best_weights=True,

verbose=1

)

reduce_lr = ReduceLROnPlateau(

monitor="val_loss",

factor=0.2,

patience=2,

verbose=1

)

In [24]:
history = model.fit(

train_generator,

validation_data=test_generator,

epochs=20,

callbacks=[

checkpoint,

earlystop,

reduce_lr

]

)

Epoch 1/20
898/898 ━━━━━━━━━━━━━━━━━━━━ 0s 403ms/step - accuracy: 0.2950 - loss: 2.0462
Epoch 1: val_accuracy improved from None to 0.39078, saving model to /content/drive/MyDrive/MoodifyAI/best_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/MoodifyAI/best_model.keras
898/898 ━━━━━━━━━━━━━━━━━━━━ 417s 431ms/step - accuracy: 0.3526 - loss: 1.8447 - val_accuracy: 0.3908 - val_loss: 1.6996 - learning_rate: 1.0000e-04
Epoch 2/20
898/898 ━━━━━━━━━━━━━━━━━━━━ 0s 378ms/step - accuracy: 0.4402 - loss: 1.5857
Epoch 2: val_accuracy improved from 0.39078 to 0.47033, saving model to /content/drive/MyDrive/MoodifyAI/best_model.keras

Epoch 2: finished saving model to /content/drive/MyDrive/MoodifyAI/best_model.keras
898/898 ━━━━━━━━━━━━━━━━━━━━ 347s 387ms/step - accuracy: 0.4548 - loss: 1.5554 - val_accuracy: 0.4703 - val_loss: 1.4958 - learning_rate: 1.0000e-04
Epoch 3/20
898/898 ━━━━━━━━━━━━━━━━━━━━ 0s 381ms/step - accuracy: 0.4951 - loss: 1.4718
Epoch 3: val_accuracy impro

In [25]:
model.save("/content/drive/MyDrive/MoodifyAI/final_model.keras")

In [26]:
from google.colab import files

files.download("/content/drive/MyDrive/MoodifyAI/best_model.keras")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [27]:
print(max(history.history["accuracy"]))
print(max(history.history["val_accuracy"]))

0.7054234147071838
0.6308164000511169


In [28]:
!ls "/content/drive/MyDrive/MoodifyAI"

best_model.keras  final_model.keras
